## Step 1: Data Loading

In [ ]:
# Image data pipeline configuration - v3
import torch
from torch.utils import data
import torchvision
from PIL import Image
import numpy as np

# Constants
IMAGE_SIZE = 224
BATCH_SZ = 16
DATA_DIR = 'dataset/'
RGB_MEANS = [0.485, 0.456, 0.406]  # ImageNet means
RGB_STDS = [0.229, 0.224, 0.225]   # ImageNet stds

# Setup preprocessing chain
transforms_list = []
transforms_list.append(torchvision.transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)))
transforms_list.append(torchvision.transforms.ToTensor())
transforms_list.append(torchvision.transforms.Normalize(RGB_MEANS, RGB_STDS))
image_transforms = torchvision.transforms.Compose(transforms_list)

# Create dataset and loader objects
face_dataset = torchvision.datasets.ImageFolder(root=DATA_DIR, transform=image_transforms)
face_dataloader = data.DataLoader(
    face_dataset, 
    batch_size=BATCH_SZ,
    shuffle=False,
    drop_last=False
)


## Step2: Feature Extraction

In [ ]:
import psycopg2
import numpy as np
from tqdm import tqdm
from psycopg2 import sql
from psycopg2.extras import execute_batch

# Connect to PostgreSQL database
db_params = {
    "dbname": "facedb", "user": "postgres", 
    "password": "postgres", "host": "localhost"
}
connection = psycopg2.connect(**db_params)
cursor = connection.cursor()

# Ensure the table exists
cursor.execute('''
    CREATE TABLE IF NOT EXISTS image_features (
        id SERIAL PRIMARY KEY,
        batch_id SMALLINT,
        image_id SMALLINT, 
        channel_id SMALLINT,
        y_pos SMALLINT,
        x_pos SMALLINT,
        pixel_value REAL
    )
''')
connection.commit()

# Extract function with batched approach
def process_images(dataloader, sample_density=0.1):
    # Calculate sampling stride
    skip = int(1.0/sample_density)
    counter = 0
    
    # Statement for insertion
    insert_stmt = sql.SQL(
        "INSERT INTO image_features "
        "(batch_id, image_id, channel_id, y_pos, x_pos, pixel_value) "
        "VALUES (%s, %s, %s, %s, %s, %s)"
    )
    
    # Process each image batch
    for batch_num, data in enumerate(tqdm(dataloader, desc="Processing")):
        images, _ = data
        imgs_np = images.numpy()
        
        # Use smaller chunks for better memory management
        records = []
        
        # Iterate through each image in batch
        for img_num in range(imgs_np.shape[0]):
            img = imgs_np[img_num]
            
            # Process channels
            for channel in range(img.shape[0]):
                # Sample pixels at regular intervals
                for row in range(0, img.shape[1], skip):
                    for col in range(0, img.shape[2], skip):
                        # Get pixel value
                        value = float(img[channel, row, col])
                        
                        # Skip negligible values to save space
                        if abs(value) <= 0.001:
                            continue
                            
                        # Add to batch
                        records.append(
                            (batch_num, img_num, channel, row, col, round(value, 4))
                        )
        
        # Use execute_batch for efficiency
        if records:
            execute_batch(cursor, insert_stmt, records, page_size=1000)
            connection.commit()
            
            # Update counter and log progress
            counter += len(records)
            print(f"Batch {batch_num}: added {len(records)} features")
    
    return counter

# Run extraction with sampling
num_features = process_images(data_loader, 0.1)
print(f"Extraction complete - stored {num_features} pixel features")

# Clean up database connection
cursor.close()
connection.close()


## Step 3: Model Profiling

In [ ]:
import onnx
import time
import numpy as np
from onnxruntime import InferenceSession

def benchmark_onnx(path, shape=(1, 3, 224, 224), iters=50):
    net = onnx.load(path)
    rt = InferenceSession(path)
    
    x = np.random.normal(size=shape).astype('float32')
    begin = time.perf_counter()
    for i in range(iters):
        rt.run(None, {rt.get_inputs()[0].name: x})
    elapsed = time.perf_counter() - begin
    
    return {
        "params": len(net.graph.initializer),
        "avg_ms": (elapsed/iters)*1000
    }

results = benchmark_onnx("resnext50.onnx")
print(f"Parameters: {results['params']}, Time: {results['avg_ms']:.2f}ms")


## Step 4: Data Preparation

In [ ]:
def __init__(self, connection_string):
    # Database connection setup
    self.connection = psycopg2.connect(connection_string)
    self.cursor = self.connection.cursor()
    self.cleanup_list = []
    
def __del__(self):
    # Auto-cleanup
    if hasattr(self, 'connection') and self.connection:
        if hasattr(self, 'cursor') and self.cursor:
            self.cursor.close()
        self.connection.close()

def execute_sql(self, statement, params=None, batch=False, data=None):
    """Execute SQL with proper error handling"""
    try:
        if batch and data:
            execute_batch(self.cursor, statement, data)
        elif params:
            self.cursor.execute(statement, params)
        else:
            self.cursor.execute(statement)
        self.connection.commit()
    except Exception as e:
        self.connection.rollback()
        print(f"SQL Error: {e}")
        raise

def analyze_model(self, model_file):
    """Load model and extract stats"""
    network = onnx.load(model_file)
    print(f"Network Structure:")
    print(f"• Total nodes: {len(network.graph.node)}")
    print(f"• Weight tensors: {len(network.graph.initializer)}")
    return network

def extract_weight_tables(self, network):
    """Convert model weights to database-friendly format"""
    tensor_table_map = {}
    
    for weight in network.graph.initializer:
        # Parse tensor info
        weight_id = weight.name
        dimensions = tuple(dim for dim in weight.dims)
        
        # Extract numeric data
        tensor_data = None
        if weight.data_type == 1:  # FLOAT
            tensor_data = np.frombuffer(weight.raw_data, dtype=np.float32).reshape(dimensions)
        else:
            continue  # Skip non-float tensors
            
        # Create table name (sanitized)
        table_id = f"wt_{weight_id.replace('.', '_').lower()}"
        tensor_table_map[table_id] = [tensor_data]
        
        # Output info for significant tensors
        if len(dimensions) > 1:
            print(f"&#10003; Weight: {weight_id} &rarr; Table: {table_id} ({dimensions})")
            
    return tensor_table_map

def import_weights_to_db(self, weight_tables):
    """Store weight tensors in database tables"""
    create_statements = []
    insert_operations = []
    drop_statements = []
    param_records = {}
    
    # Process each weight tensor
    for table_name, tensor_list in weight_tables.items():
        tensor = tensor_list[0]
        shape = tensor.shape
        
        # Handle different tensor dimensions
        if len(shape) == 2:  # 2D weights (fc layers, etc)
            create_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (output_unit SMALLINT, input_unit SMALLINT, weight_value REAL)"
            insert_sql = f"INSERT INTO {table_name} (output_unit, input_unit, weight_value) VALUES (%s, %s, %s)"
            
            # Flatten the tensor for insertion
            records = []
            for i in range(shape[0]):
                for j in range(shape[1]):
                    records.append((i, j, float(tensor[i, j])))
                    
        else:  # 3D/4D weights (conv layers, etc)
            create_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (filter_group SMALLINT, filter_id SMALLINT, element_id SMALLINT, weight_value REAL)" 
            insert_sql = f"INSERT INTO {table_name} (filter_group, filter_id, element_id, weight_value) VALUES (%s, %s, %s, %s)"
            
            # Flatten the multi-dimensional tensor
            records = []
            for g in range(shape[0]):
                for f in range(shape[1]):
                    for e in range(shape[2]):
                        records.append((g, f, e, float(tensor[g, f, e])))
        
        # Store for batch execution
        param_records[table_name] = records
        create_statements.append(create_sql)
        insert_operations.append((insert_sql, table_name))
        drop_statements.append(f"DROP TABLE IF EXISTS {table_name};")
    
    # Execute all create statements
    for stmt in create_statements:
        self.execute_sql(stmt)
    
    # Execute all insert operations
    for op in insert_operations:
        self.execute_sql(op[0], batch=True, data=param_records[op[1]])
        
    # Save drop statements for cleanup
    with open("model_cleanup.sql", 'w') as f:
        f.write('\n'.join(drop_statements))

def create_conv_mapping(self, name, input_size, kernel, stride, pad, channels):
    """Create mapping tables for convolution operations"""
    if name in ["existing_mapping1", "existing_mapping2"]:
        return  # Skip existing mappings
        
    h, w = input_size, input_size
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {name} (
        channel_id SMALLINT, 
        input_pos SMALLINT, 
        output_pos SMALLINT, 
        kernel_pos SMALLINT
    )
    """
    self.execute_sql(create_sql)
    
    # Generate mapping data
    mappings = []
    output_idx = 0
    
    for y_out in range(0, h + 2*pad - kernel + 1, stride):
        for x_out in range(0, w + 2*pad - kernel + 1, stride):
            kernel_idx = 0
            for c in range(channels):
                for ky in range(kernel):
                    for kx in range(kernel):
                        y_in = y_out + ky
                        x_in = x_out + kx
                        
                        if y_in >= pad and y_in < h+pad and x_in >= pad and x_in < w+pad:
                            input_idx = (y_in-pad) * w + (x_in-pad)
                            mappings.append((c, input_idx, output_idx, kernel_idx))
                        kernel_idx += 1
            output_idx += 1
    
    # Batch insert the mappings
    insert_sql = f"INSERT INTO {name} (channel_id, input_pos, output_pos, kernel_pos) VALUES (%s, %s, %s, %s)"
    self.execute_sql(insert_sql, batch=True, data=mappings)
    self.cleanup_list.append(name)

def create_grouped_conv_mapping(self, name, input_size, kernel, stride, pad, channels, groups):
    """Create mapping tables for grouped convolution operations"""
    skip_list = ["skip_map1", "skip_map2", "skip_map3", "skip_map4", 
                 "skip_map5", "skip_map6", "skip_map7", "skip_map8", "skip_map9"]
    if name in skip_list:
        return
        
    h = w = input_size
    
    # Create mapping table
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {name} (
        group_id SMALLINT,
        channel_id SMALLINT, 
        input_pos SMALLINT, 
        output_pos SMALLINT, 
        kernel_pos SMALLINT
    )
    """
    self.execute_sql(create_sql)
    
    # Calculate channel group assignments
    channels_per_group = channels // groups
    channel_groups = [c // channels_per_group for c in range(channels)]
    
    # Generate mapping data
    mappings = []
    output_idx = 0
    kernel_positions_per_group = kernel * kernel * (channels // groups)
    
    for y_out in range(0, h + 2*pad - kernel + 1, stride):
        for x_out in range(0, w + 2*pad - kernel + 1, stride):
            pos = 0
            for c in range(channels):
                for ky in range(kernel):
                    for kx in range(kernel):
                        y_in = y_out + ky
                        x_in = x_out + kx
                        
                        if y_in >= pad and y_in < h+pad and x_in >= pad and x_in < w+pad:
                            input_idx = (y_in-pad) * w + (x_in-pad)
                            group_id = channel_groups[c]
                            mappings.append((
                                group_id, 
                                c, 
                                input_idx, 
                                output_idx, 
                                pos % kernel_positions_per_group
                            ))
                        pos += 1
            output_idx += 1
            
    # Batch insert the mappings
    insert_sql = f"""
    INSERT INTO {name} 
    (group_id, channel_id, input_pos, output_pos, kernel_pos) 
    VALUES (%s, %s, %s, %s, %s)
    """
    self.execute_sql(insert_sql, batch=True, data=mappings)
    self.cleanup_list.append(name)

def transform_model(self, model_path):
    """Main function to transform ONNX model to DB tables"""
    start = time.time()
    
    # Load and analyze model
    print(f"Processing model: {model_path}")
    network = self.analyze_model(model_path)
    
    # Extract weights as tables
    weight_tables = self.extract_weight_tables(network)
    print(f"Extracted {len(weight_tables)} weight tables")
    
    # Import weights to database 
    self.import_weights_to_db(weight_tables)
    
    # Create conv layer mappings
    self.create_conv_mapping("input_conv_map", 224, 7, 2, 3, 3)
    self.create_conv_mapping("block1_conv_map", 56, 3, 1, 1, 64)
    

## Step 5: Query Composition

In [ ]:
'''
WITH
conv201_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM input_feature_map F
  INNER JOIN weight_conv201 K
    ON F.order_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.matrix_id
),
relu203_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv201_fwd0
),
mapping_relu203 AS (
  SELECT 
    batch_id, 
    F.kernel_id, 
    matrix_id, 
    order_id, 
    value
  FROM relu203_fwd0 F
  INNER JOIN mapping_relu203 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
maxpool205_fwd0 AS (
  SELECT 
    batch_id, 
    kernel_id, 
    matrix_id AS tuple_id, 
    MAX(value) AS value
  FROM mapping_relu203
  GROUP BY batch_id, kernel_id, matrix_id
),
conv206_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM maxpool205_fwd0 F
  INNER JOIN weight_conv206 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
conv207_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM maxpool205_fwd0 F
  INNER JOIN weight_conv207 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu208_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv206_fwd0
),
mapping_relu208 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu208_fwd0 F
  INNER JOIN mapping_relu208 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv209_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 4 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu208 F
  INNER JOIN weight_conv209 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu210_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv209_fwd0
),
conv211_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu210_fwd0 F
  INNER JOIN weight_conv211 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add212_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM conv207_fwd0 A
  INNER JOIN conv211_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu213_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add212_fwd0
),
conv214_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu213_fwd0 F
  INNER JOIN weight_conv214 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu215_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv214_fwd0
),
mapping_relu215 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu215_fwd0 F
  INNER JOIN mapping_relu215 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv216_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 4 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu215 F
  INNER JOIN weight_conv216 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu217_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv216_fwd0
),
conv218_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu217_fwd0 F
  INNER JOIN weight_conv218 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add219_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu213_fwd0 A
  INNER JOIN conv218_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu220_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add219_fwd0
),
conv221_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu220_fwd0 F
  INNER JOIN weight_conv221 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu222_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv221_fwd0
),
mapping_relu222 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu222_fwd0 F
  INNER JOIN mapping_relu222 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv223_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 4 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu222 F
  INNER JOIN weight_conv223 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu224_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv223_fwd0
),
conv225_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu224_fwd0 F
  INNER JOIN weight_conv225 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add226_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu220_fwd0 A
  INNER JOIN conv225_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu227_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add226_fwd0
),
conv228_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu227_fwd0 F
  INNER JOIN weight_conv228 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu229_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv228_fwd0
),
mapping_relu229 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu229_fwd0 F
  INNER JOIN mapping_relu229 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv230_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 4 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu229 F
  INNER JOIN weight_conv230 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu231_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv230_fwd0
),
conv232_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu231_fwd0 F
  INNER JOIN weight_conv232 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add233_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu227_fwd0 A
  INNER JOIN conv232_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu234_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add233_fwd0
),
conv235_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu234_fwd0 F
  INNER JOIN weight_conv235 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu236_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv235_fwd0
),
mapping_relu236 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu236_fwd0 F
  INNER JOIN mapping_relu236 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv237_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 4 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu236 F
  INNER JOIN weight_conv237 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu238_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv237_fwd0
),
conv239_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu238_fwd0 F
  INNER JOIN weight_conv239 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add240_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu234_fwd0 A
  INNER JOIN conv239_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu241_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add240_fwd0
),
conv242_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu241_fwd0 F
  INNER JOIN weight_conv242 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
mapping_relu241 AS (
  SELECT 
    batch_id, 
    matrix_id, 
    order_id, 
    value
  FROM relu241_fwd0 F
  INNER JOIN mapping_relu241 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv243_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu241 F
  INNER JOIN weight_conv243 K
    ON F.order_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.matrix_id
),
relu244_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv242_fwd0
),
mapping_relu244 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu244_fwd0 F
  INNER JOIN mapping_relu244 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv245_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 8 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu244 F
  INNER JOIN weight_conv245 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu246_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv245_fwd0
),
conv247_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu246_fwd0 F
  INNER JOIN weight_conv247 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add248_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM conv243_fwd0 A
  INNER JOIN conv247_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu249_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add248_fwd0
),
conv250_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu249_fwd0 F
  INNER JOIN weight_conv250 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu251_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv250_fwd0
),
mapping_relu251 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu251_fwd0 F
  INNER JOIN mapping_relu251 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv252_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 8 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu251 F
  INNER JOIN weight_conv252 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu253_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv252_fwd0
),
conv254_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu253_fwd0 F
  INNER JOIN weight_conv254 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add255_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu249_fwd0 A
  INNER JOIN conv254_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu256_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add255_fwd0
),
conv257_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu256_fwd0 F
  INNER JOIN weight_conv257 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu258_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv257_fwd0
),
mapping_relu258 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu258_fwd0 F
  INNER JOIN mapping_relu258 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv259_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 8 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu258 F
  INNER JOIN weight_conv259 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu260_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv259_fwd0
),
conv261_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu260_fwd0 F
  INNER JOIN weight_conv261 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add262_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu256_fwd0 A
  INNER JOIN conv261_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu263_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add262_fwd0
),
conv264_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu263_fwd0 F
  INNER JOIN weight_conv264 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu265_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv264_fwd0
),
mapping_relu265 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu265_fwd0 F
  INNER JOIN mapping_relu265 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv266_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 8 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu265 F
  INNER JOIN weight_conv266 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu267_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv266_fwd0
),
conv268_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu267_fwd0 F
  INNER JOIN weight_conv268 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add269_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu263_fwd0 A
  INNER JOIN conv268_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu270_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add269_fwd0
),
conv271_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu270_fwd0 F
  INNER JOIN weight_conv271 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu272_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv271_fwd0
),
mapping_relu272 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu272_fwd0 F
  INNER JOIN mapping_relu272 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv273_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 8 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu272 F
  INNER JOIN weight_conv273 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu274_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv273_fwd0
),
conv275_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu274_fwd0 F
  INNER JOIN weight_conv275 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add276_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu270_fwd0 A
  INNER JOIN conv275_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu277_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add276_fwd0
),
conv278_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu277_fwd0 F
  INNER JOIN weight_conv278 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
mapping_relu277 AS (
  SELECT 
    batch_id, 
    matrix_id, 
    order_id, 
    value
  FROM relu277_fwd0 F
  INNER JOIN mapping_relu277 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv279_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu277 F
  INNER JOIN weight_conv279 K
    ON F.order_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.matrix_id
),
relu280_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv278_fwd0
),
mapping_relu280 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu280_fwd0 F
  INNER JOIN mapping_relu280 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv281_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 16 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu280 F
  INNER JOIN weight_conv281 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu282_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv281_fwd0
),
conv283_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu282_fwd0 F
  INNER JOIN weight_conv283 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add284_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM conv279_fwd0 A
  INNER JOIN conv283_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu285_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add284_fwd0
),
conv286_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu285_fwd0 F
  INNER JOIN weight_conv286 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu287_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv286_fwd0
),
mapping_relu287 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu287_fwd0 F
  INNER JOIN mapping_relu287 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv288_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 16 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu287 F
  INNER JOIN weight_conv288 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu289_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv288_fwd0
),
conv290_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu289_fwd0 F
  INNER JOIN weight_conv290 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add291_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu285_fwd0 A
  INNER JOIN conv290_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu292_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add291_fwd0
),
conv293_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu292_fwd0 F
  INNER JOIN weight_conv293 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu294_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv293_fwd0
),
mapping_relu294 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu294_fwd0 F
  INNER JOIN mapping_relu294 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv295_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 16 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu294 F
  INNER JOIN weight_conv295 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu296_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv295_fwd0
),
conv297_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu296_fwd0 F
  INNER JOIN weight_conv297 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add298_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu292_fwd0 A
  INNER JOIN conv297_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu299_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add298_fwd0
),
conv300_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu299_fwd0 F
  INNER JOIN weight_conv300 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu301_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv300_fwd0
),
mapping_relu301 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu301_fwd0 F
  INNER JOIN mapping_relu301 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv302_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 16 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu301 F
  INNER JOIN weight_conv302 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu303_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv302_fwd0
),
conv304_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu303_fwd0 F
  INNER JOIN weight_conv304 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add305_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu299_fwd0 A
  INNER JOIN conv304_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu306_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add305_fwd0
),
conv307_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu306_fwd0 F
  INNER JOIN weight_conv307 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
relu308_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv307_fwd0
),
mapping_relu308 AS (
  SELECT 
    batch_id, 
    gn, 
    matrix_id, 
    order_id, 
    value
  FROM relu308_fwd0 F
  INNER JOIN mapping_relu308 M
    ON F.kernel_id = M.kernel_id AND F.tuple_id = M.tuple_id
),
conv309_fwd0 AS (
  SELECT 
    batch_id, 
    (F.gn * 16 + kernel_id) AS kernel_id, 
    F.matrix_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM mapping_relu308 F
  INNER JOIN weight_conv309 K
    ON F.order_id = K.order_id AND F.gn = K.gn
  GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
),
relu310_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM conv309_fwd0
),
conv311_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id, 
    F.tuple_id AS tuple_id,
    SUM(F.value * K.value) AS value
  FROM relu310_fwd0 F
  INNER JOIN weight_conv311 K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id, F.tuple_id
),
add312_fwd0 AS (
  SELECT 
    A.batch_id, 
    A.kernel_id, 
    A.tuple_id,
    A.value + B.value AS value
  FROM relu306_fwd0 A
  INNER JOIN conv311_fwd0 B
    ON A.batch_id = B.batch_id 
    AND A.kernel_id = B.kernel_id 
    AND A.tuple_id = B.tuple_id
),
relu313_fwd0 AS (
  SELECT batch_id, kernel_id, tuple_id, GREATEST(value, 0) AS value
  FROM add312_fwd0
),
averagepool315_fwd0 AS (
  SELECT 
    batch_id, 
    kernel_id, 
    AVG(value) AS value
  FROM relu313_fwd0
  GROUP BY batch_id, kernel_id
),
matmul318_fwd0 AS (
  SELECT 
    batch_id, 
    K.kernel_id,
    SUM(F.value * K.value) AS value
  FROM averagepool315_fwd0 F
  INNER JOIN transpose317_input K
    ON F.kernel_id = K.order_id
  GROUP BY F.batch_id, K.kernel_id
)
SELECT l.name AS res
FROM cifar10_labels l
JOIN (
  SELECT DISTINCT ON (batch_id) 
    batch_id, 
    kernel_id + 1 AS label
  FROM matmul318_fwd0
  ORDER BY batch_id, value DESC
) t
  ON t.label = l.label;
'''